# Hidden-set predictions for Kryptonite-20, -30 and -40

For each $n$ we compare three models on validation data: the MLP H1 chose, and the paper's polynomial LR and the lookup
table on the top-$k$ features ($k$ and $d$ as chosen in `H1_features`). The best one makes the hidden-set predictions.
The polynomial LR and the lookup table are refitted on all labelled data; the MLP keeps its validation split for early
stopping. Kryptonite-40-Adapt's predictions come from the LoRA notebook.

In [ ]:
import subprocess
import sys

import numpy as np
import torch
from sklearn.model_selection import train_test_split

from include.config import NS, HIDDENLABELS
from include.data import load, load_hidden, to_sign
from include.models import mlp, fit, predict_logits
from include.features import flip_influence, fit_poly, fit_lookup
from include.tracking import track, save_json, load_json

torch.set_num_threads(4)
h1, hf = load_json("H1_learnability.json"), load_json("H1_features.json")
assert h1 is not None and hf is not None, "run H1_learnability.ipynb and H1_features.ipynb first"
HIDDENLABELS.mkdir(exist_ok=True)

In [ ]:
final = {}
with track("final_predictions"):
    for n in NS:
        X, y = load(n)
        X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
        S_tr, S_va, S_all = to_sign(X_tr), to_sign(X_va), to_sign(X)

        cfg = h1["chosen"][str(n)]["MLP (sign x)"]
        torch.manual_seed(0)
        model, _ = fit(mlp(n, cfg["width"], cfg["depth"]), S_tr, y_tr, S_va, y_va, weight_decay=cfg["weight_decay"], seed=0)
        order = np.argsort(flip_influence(model, S_tr))[::-1]
        k, d = map(int, hf["summary"][str(n)]["poly"]["choice"].split(","))
        k_look = int(hf["summary"][str(n)]["lookup"]["choice"])

        candidates = {
            "MLP": lambda S: (predict_logits(model, S) > 0).astype(int),
            f"poly LR (k={k}, d={d})": fit_poly(S_tr, y_tr, order[:k], d),
            f"lookup table (k={k_look})": fit_lookup(S_tr, y_tr, order[:k_look]),
        }
        val = {name: float((p(S_va) == y_va).mean()) for name, p in candidates.items()}
        best = max(val, key=val.get)
        if best.startswith("poly"):
            predict = fit_poly(S_all, y, order[:k], d)
        elif best.startswith("lookup"):
            predict = fit_lookup(S_all, y, order[:k_look])
        else:
            predict = candidates["MLP"]

        y_hidden = np.asarray(predict(to_sign(load_hidden(n))), dtype=np.int64)
        np.save(HIDDENLABELS / f"y_predicted_{n}.npy", y_hidden)
        final[n] = {"val": val, "used": best, "top_features": order[:12].tolist(), "share_1": float(y_hidden.mean())}
        print(f"n={n}: " + ", ".join(f"{name} {a:.3f}" for name, a in val.items()) + f"  ->  using {best}", flush=True)

save_json({str(n): v for n, v in final.items()}, "final_predictions.json")

Check the format with the coursework's own checker (`y_predicted_40_adapt.npy` comes from the LoRA notebook, so it
is reported missing until that has run).

In [ ]:
print(subprocess.run([sys.executable, "../check_submission.py", ".."], capture_output=True, text=True).stdout)